In [3]:
import pandas as pd

import sqlite3

 

# Create an in-memory SQLite database connection

conn = sqlite3.connect(':memory:')

cursor = conn.cursor()

 

print("Setup complete. Environment ready.")

 

# Define a messy, unnormalized dataset (0NF)

# Issues: Non-atomic values (multiple text books), redundant data, mixed concerns.

data_0nf = {

    "Student_ID": [101, 101, 102, 103, 104],

    "Student_Name": ["Alice", "Alice", "Bob", "Charlie", "David"],

    "Course_Code": ["CS101", "MATH201", "CS101", "CS102", "MATH201"],

    "Course_Name": ["Intro to CS", "Calculus I", "Intro to CS", "Data Structures", "Calculus I"],

    "Instructor_ID": ["INS_01", "INS_02", "INS_01", "INS_03", "INS_05"],

    "Instructor_Name": ["Dr. Smith", "Dr. Jones", "Dr. Smith", "Dr. Alan", "Dr. Jones"],

    "Instructor_Office": ["Room 401", "Room 502", "Room 401", "Room 401", "Room 502"],

    "Textbooks_Required": ["Python Basics, Intro to CLI", "Calculus Vol 1", "Python Basics, Intro to CLI", "Algorithms Vol 1", "Calculus Vol 1"],

    "Grade": ["A", "B", "A", "B+", "A-"]

}

 

df_0nf = pd.DataFrame(data_0nf)

df_0nf.to_sql('Unnormalized_Leasing', conn, index=False, if_exists='replace')

 

print("\n--- Unnormalized Data (0NF) ---")

df_0nf

 
#0NF dummy data

 
# Flattening the Textbooks_Required column to ensure atomicity

df_1nf = df_0nf.assign(Textbooks_Required=df_0nf['Textbooks_Required'].str.split(', ')).explode('Textbooks_Required')

 

# Save to SQL

df_1nf.to_sql('Table_1NF', conn, index=False, if_exists='replace')

 

print("--- 1NF Data (Atomic values enforced) ---")

print(f"Row count increased from {len(df_0nf)} to {len(df_1nf)} due to flattening.")

df_1nf

 
#code for doing first normalization on the table

 

Setup complete. Environment ready.

--- Unnormalized Data (0NF) ---
--- 1NF Data (Atomic values enforced) ---
Row count increased from 5 to 7 due to flattening.


,Student_ID,Student_Name,Course_Code,Course_Name,Instructor_ID,Instructor_Name,Instructor_Office,Textbooks_Required,Grade
0,101,Alice,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Python Basics,A
0,101,Alice,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Intro to CLI,A
1,101,Alice,MATH201,Calculus I,INS_02,Dr. Jones,Room 502,Calculus Vol 1,B
2,102,Bob,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Python Basics,A
2,102,Bob,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Intro to CLI,A
3,103,Charlie,CS102,Data Structures,INS_03,Dr. Alan,Room 401,Algorithms Vol 1,B+
4,104,David,MATH201,Calculus I,INS_05,Dr. Jones,Room 502,Calculus Vol 1,A-


In [9]:
# Extract Instructors into a separate table to eliminate transitive dependency

df_instructors_3nf = df_courses_2nf[['Instructor_ID', 'Instructor_Name', 'Instructor_Office']].drop_duplicates().reset_index(drop=True)

 

# Clean up Courses table to only keep the Instructor_ID as a Foreign Key

df_courses_3nf = df_courses_2nf[['Course_Code', 'Course_Name', 'Instructor_ID']].drop_duplicates().reset_index(drop=True)

 

# Save to SQL

df_instructors_3nf.to_sql('Instructors_3NF', conn, index=False, if_exists='replace')

df_courses_3nf.to_sql('Courses_3NF', conn, index=False, if_exists='replace')

 

print("--- 3NF Schema Refinement ---")

print("\n[Courses_3NF] (No transitive dependency):")

display(df_courses_3nf)

print("\n[Instructors_3NF] (New isolated table):")

display(df_instructors_3nf)

 
#For removing the transitive dependencies

NameError: name 'df_courses_2nf' is not defined